In [0]:
### Importando bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [0]:
df_cadastro_EDA_01 = spark.read.parquet(
    '/Volumes/hackathon_2025/default/source/base_dados_cadastrais/'
)   
display(df_cadastro_EDA_01)

In [0]:
# verificando o tipo dos dados

df_cadastro_EDA_01.dtypes

In [0]:
#listando as colunas
list(df_cadastro_EDA_01.columns)

In [0]:
# quantas linhas tem o dataframe
num_rows = df_cadastro_EDA_01.count() 
num_cols = len(df_cadastro_EDA_01.columns)
print(f"Rows: {num_rows}, Columns: {num_cols}")

In [0]:
# Calculando a idade e criando faixas etárias

from pyspark.sql import functions as F

# Corrigindo o tipo da coluna DATADENASCIMENTO para Date
# Calculando a idade
# Usa F.to_date para converter string 'dd/MM/yyyy' para Date

# Adiciona coluna de data convertida
# (alternativamente, pode-se fazer inline, mas aqui é mais claro)
df_cadastro_EDA_01 = df_cadastro_EDA_01.withColumn(
    "DATADENASCIMENTO_DATE", F.to_date(F.col("DATADENASCIMENTO"), "dd/MM/yyyy")
)
df_cadastro_EDA_01 = df_cadastro_EDA_01.withColumn(
    "IDADE", F.floor(F.datediff(F.current_date(), F.col("DATADENASCIMENTO_DATE")) / 365.25)
)

# Criando faixas etárias
df_segmentacao = df_cadastro_EDA_01.withColumn(
    "FAIXA_ETARIA",
    F.when(F.col("IDADE") < 25, "18-24")
     .when((F.col("IDADE") >= 25) & (F.col("IDADE") <= 40), "25-40")
     .when((F.col("IDADE") >= 41) & (F.col("IDADE") <= 60), "41-60")
     .otherwise("60+")
)

# Analisando Migração por Faixa Etária e Safra
df_migracao = df_segmentacao.groupBy("SAFRA", "FAIXA_ETARIA").agg(
    
    F.count("NUM_CPF").alias("volume_vendas")
)

display(df_migracao)


Databricks visualization. Run in Databricks to view.

In [0]:
display(df_cadastro_EDA_01)

In [0]:
df_cadastro_EDA_01.registerTempTable('df')

In [0]:
df_segmentacao.registerTempTable('df_segmentacao')


In [0]:
%sql
select safra,
      idade,
      count(*) as qtd
from df
where FPD is not null and flag_mig2 = "PRE"
group by safra, idade
order by safra, idade

In [0]:
%sql
select * 
from df
where (idade is null or idade < 18) and
 FPD is not null and flag_mig2 = "PRE"

In [0]:
%sql
select safra,
      avg(idade)
      
from df
where FPD is not null and flag_mig2 = "PRE" and FPD = 1

group by safra
order by safra

In [0]:
%sql
select * from df_segmentacao

In [0]:
%sql
SELECT SAFRA,
       FAIXA_ETARIA,
       ROUND(SUM(FPD)/COUNT(*),2) AS PCT_FPD,
       COUNT(*) AS QTD
FROM df_segmentacao
WHERE FPD IS NOT NULL  AND flag_mig2 = 'PRE'
GROUP BY SAFRA, FAIXA_ETARIA
ORDER BY SAFRA, faixa_etaria

Databricks visualization. Run in Databricks to view.

FPD por CEP_3, Estado e Região


In [0]:
import pandas as pd

# Tabela de referência de faixas de CEP
cep_ref = pd.DataFrame({
    "cep_inicio": [100,200,290,300,400,490,500,570,580,590,600,640,650,
                   660,689,690,693,694,699,700,737,768,770,780,789,790,
                   800,880,900],
    "cep_fim":   [199,289,299,399,489,499,569,579,589,599,639,649,659,
                   688,689,692,693,698,699,736,767,769,779,788,789,799,
                   879,899,999],
    "Estado": [
        "São Paulo","Rio de Janeiro","Espírito Santo","Minas Gerais",
        "Bahia","Sergipe","Pernambuco","Alagoas","Paraíba",
        "Rio Grande do Norte","Ceará","Piauí","Maranhão",
        "Pará","Amapá","Amazonas","Roraima","Amazonas","Acre",
        "Distrito Federal","Goiás","Rondônia","Tocantins",
        "Mato Grosso","Rondônia","Mato Grosso do Sul",
        "Paraná","Santa Catarina","Rio Grande do Sul"
    ],
    "Regiao": [
        "Sudeste","Sudeste","Sudeste","Sudeste",
        "Nordeste","Nordeste","Nordeste","Nordeste","Nordeste",
        "Nordeste","Nordeste","Nordeste","Nordeste",
        "Norte","Norte","Norte","Norte","Norte","Norte",
        "Centro-Oeste","Centro-Oeste","Norte","Norte",
        "Centro-Oeste","Norte","Centro-Oeste",
        "Sul","Sul","Sul"
    ]
})

def mapear_estado_regiao(cep3, tabela_ref):
    linha = tabela_ref[
        (tabela_ref["cep_inicio"] <= cep3) &
        (tabela_ref["cep_fim"] >= cep3)
    ]
    
    if linha.empty:
        return pd.Series(["Desconhecido", "Desconhecida"])
    
    return linha.iloc[0][["Estado", "Regiao"]]

   

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when

# Usando o nome correto da coluna: 'CEP_3_digitos'
df_cadastro_EDA_01 = df_cadastro_EDA_01.withColumn('regiao',
    when((col('CEP_3_digitos') >= 10) & (col('CEP_3_digitos') <= 199), 'SP - São Paulo')
    .when((col('CEP_3_digitos') >= 200) & (col('CEP_3_digitos') <= 289), 'RJ - Rio de Janeiro')
    .when((col('CEP_3_digitos') >= 290) & (col('CEP_3_digitos') <= 299), 'ES - Espírito Santo')
    .when((col('CEP_3_digitos') >= 300) & (col('CEP_3_digitos') <= 399), 'MG - Minas Gerais')
    .when((col('CEP_3_digitos') >= 400) & (col('CEP_3_digitos') <= 499), 'PR - Paraná')
    .when((col('CEP_3_digitos') >= 500) & (col('CEP_3_digitos') <= 599), 'SC - Santa Catarina')
    .when((col('CEP_3_digitos') >= 600) & (col('CEP_3_digitos') <= 699), 'RS - Rio Grande do Sul')
    .when((col('CEP_3_digitos') >= 700) & (col('CEP_3_digitos') <= 727), 'DF - Distrito Federal')
    .when((col('CEP_3_digitos') >= 728) & (col('CEP_3_digitos') <= 769), 'GO - Goiás')
    .when((col('CEP_3_digitos') >= 770) & (col('CEP_3_digitos') <= 779), 'TO - Tocantins')
    .when((col('CEP_3_digitos') >= 780) & (col('CEP_3_digitos') <= 799), 'MT/RO - Mato Grosso/Rondônia')
    .when((col('CEP_3_digitos') >= 800) & (col('CEP_3_digitos') <= 879), 'MS - Mato Grosso do Sul')
    .when((col('CEP_3_digitos') >= 880) & (col('CEP_3_digitos') <= 999), 'Norte - Região Norte')
    .otherwise('CEP Inválido')
)



In [0]:
display(df_cadastro_EDA_01)


In [0]:
df_cadastro_EDA_01.registerTempTable('cadastro_EDA_01')

In [0]:
%sql
SELECT SAFRA,
       regiao,
       ROUND(SUM(FPD)/COUNT(*),2) AS PCT_FPD,
       COUNT(*) AS QTD
FROM cadastro_EDA_01
WHERE FPD IS NOT NULL  AND flag_mig2 = 'PRE'
GROUP BY SAFRA, regiao
ORDER BY SAFRA, regiao

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT CEP_3_digitos,
       COUNT(FPD) AS volume,
       ROUND(AVG(FPD), 2) AS taxa_fpd
FROM df
GROUP BY CEP_3_digitos

In [0]:
df_cadastro_EDA_01_pd = df_cadastro_EDA_01.toPandas()

In [0]:


fpd_cep = (
    df_cadastro_EDA_01_pd.groupby('CEP_3_digitos')
      .agg(volume=('FPD', 'count'),
           taxa_fpd=('FPD', 'mean'))
      .reset_index()
)

In [0]:
# Filtrar CEPs relevantes (volume mínimo)
fpd_cep_filtro = fpd_cep[fpd_cep['volume'] >= 500]

plt.figure(figsize=(8, 5))
plt.scatter(fpd_cep_filtro['volume'], fpd_cep_filtro['taxa_fpd'])
plt.xlabel('Volume de Clientes')
plt.ylabel('Taxa de FPD')
plt.title('Volume vs Taxa de FPD por CEP_3')
plt.grid(True)
plt.tight_layout()
plt.show()

In [0]:
fpd_estado = (
    df_cadastro_EDA_01_pd.groupby('Estado')
      .agg(volume=('FPD', 'count'),
           taxa_fpd=('FPD', 'mean'))
      .reset_index()
)

In [0]:
plt.figure(figsize=(8, 5))
plt.scatter(fpd_estado['volume'], fpd_estado['taxa_fpd'])
plt.xlabel('Volume de Clientes')
plt.ylabel('Taxa de FPD')
plt.title('Volume vs Taxa de FPD por Estado')
plt.grid(True)
plt.tight_layout()
plt.show()

In [0]:
fpd_regiao = (
    df_cadastro_EDA_01_pd.groupby('Regiao')
      .agg(volume=('FPD', 'count'),
           taxa_fpd=('FPD', 'mean'))
      .reset_index()
)

In [0]:
plt.figure(figsize=(8, 5))
plt.scatter(fpd_regiao['volume'], fpd_cep_filtro['taxa_fpd'])
plt.xlabel('Volume de Clientes')
plt.ylabel('Taxa de FPD')
plt.title('Volume vs Taxa de FPD por Região')
plt.grid(True)
plt.tight_layout()
plt.show()